In [1]:
import os
import re
import csv
from datetime import datetime

In [2]:
source = r"G:\c_out\Call 09-05\Pages and Pics\Call Editorial-Pics\CALL COPY SEPT. 5"
destination = r"G:\c_out2"


In [3]:
def extract_quark_text(file_path):
    """Identifies QuarkXPress 3 files and extracts coherent text blocks."""
    try:
        with open(file_path, 'rb') as f:
            # Read only the first 1024 bytes to check signature for speed
            header_data = f.read(1024)
            
            # Change: Search for signature within the first 100 bytes 
            # instead of a strict startswith check.
            if b'MMXPR3' not in header_data[:100]:
                return None

            # Re-read full file for text extraction
            f.seek(0)
            data = f.read()

        # Extract printable sequences (MacRoman/ASCII) at least 15 chars long
        text_blobs = re.findall(b'[\x20-\x7E\xA0-\xFF]{15,}', data)
        
        clean_text = []
        skip_keywords = ['LaserWriter', 'Century-Book', 'QRGB', 'QCMK', 'PnHx', 'TEXT', 'JXQAA']

        for blob in text_blobs:
            try:
                decoded = blob.decode('mac_roman').strip()
                if not any(k in decoded for k in skip_keywords):
                    clean_text.append(decoded)
            except UnicodeDecodeError:
                continue
                
        return "\n\n".join(clean_text)
    except Exception as e:
        return f"Error: {e}"

def run_inventory(source_dir, output_dir):
    """Processes files, creates individual TXTs, and one master CSV."""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    csv_path = os.path.join(output_dir, "inventory_results.csv")
    
    with open(csv_path, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['file_name', 'folder', 'date_modified', 'size_kb', 'extracted_text']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for root, dirs, files in os.walk(source_dir):
            for filename in files:
                file_path = os.path.join(root, filename)
                
                # Metadata
                stats = os.stat(file_path)
                dt_mod = datetime.fromtimestamp(stats.st_mtime).strftime('%Y-%m-%d %H:%M:%S')
                size_kb = round(stats.st_size / 1024, 2)
                
                content = extract_quark_text(file_path)
                
                if content:
                    txt_name = f"{filename}.txt"
                    with open(os.path.join(output_dir, txt_name), 'w', encoding='utf-8') as tf:
                        tf.write(content)
                    
                    writer.writerow({
                        'file_name': filename,
                        'folder': os.path.basename(root),
                        'date_modified': dt_mod,
                        'size_kb': size_kb,
                        'extracted_text': content[:1000] # Increased preview size
                    })
                    print(f"Success: {filename}")
                else:
                    print(f"Skipped: {filename} (No Quark signature found)")


In [4]:
#Set your paths here

run_inventory(source, destination)

Success: gide-- Let’s Consider
Success: Spt - Three-peat (Westport )
Success: Spt - Summer (NFL Coaches)
Success: Spt - Seniors (Southeast)
Success: Spt - Offensive Line (Chiefs)
Success: Spt - New Coach (Central)
Success: Spt - NHL Report (Diversity)
Success: Spt - NHL Report
Success: Spt - NCAA Raises
Success: Spt - Monday Niters 9_5
Success: Spt - Lincoln U. (Hoops)
Success: Spt - Brazil's Supremacy
Success: Spt - A Rarity (Team Doctor)
Success: Spt - 3-col We've Got The Beef
Success: Spt - 3-col Knights Looking
Success: Spt - 3-col Can Westport
Success: Spt - 1-col Fast Man At Central
Success: S-Socially Speaking
Success: S-3 col Hilltop Saddle Club
Success: S-3 col Annal Black Open
Success: S--Cooking
Success: S- National Beauty (MUST)
Success: S- 2 col. At National
Success: S - Hometown Hero (MUST)
Success: S - Club Notes 9_5
Success: S - 4-col Celebrating African
Success: S - 3-col Ladies Of Link
Success: S - 2-col Paying Tribute
Success: NNPA--Frank Bolden Dead
Success: NNPA Ne